# JP Morgan Quant V4 — Dual Attention Transformer

缺陷 1（Critical）：Cross-Stock Attention 是假的
你的 DualBlock 里，所谓的"cross-sectional"其实只是对 batch 内所有样本做了个 mean，然后用一个 gate 去调制。问题在于你的 DataLoader 是 shuffle=True 的——同一个 batch 里可能混合了不同日期、不同股票的样本，这个 batch mean 根本不代表"同一天的市场状态"。
MASTER 的做法是真正在同一时间步上把所有股票的 embedding 拼在一起做 multi-head attention arXiv，让模型学到"今天 NVDA 在涨，AMD 也在涨，这是板块联动"这样的信号。它同时建模了 momentary correlation（同一时刻的跨股票关系）和 cross-time correlation（不同时刻的跨股票关系） arXiv。
要实现真正的 cross-sectional attention，你的数据组织方式需要彻底改变——从"逐样本"变成"逐日期"，每个 batch 是同一天所有股票的数据。

缺陷 2（Critical）：数据规模严重不足
你有 ~60 只股票、9 年数据，大约 100K 样本。Quantformer 那篇论文收集了超过 4600 只股票、超过 500 万条滚动数据进行训练 arXiv。MASTER 使用的是 CSI300 和 CSI800，训练集跨度从 2008 年到 2020 年 AAAI。
你的模型有 ~800K 参数但只有 100K 样本，参数-样本比接近 1:100，对于金融这种低信噪比数据来说严重不足。一般经验是至少 1:1000。解决方案是扩大股票池到 S&P 500 全部成分股，或者大幅缩小模型。

缺陷 3（Critical）：没有 Validation Set，也没有 Rolling Window
你用的是一刀切的 70/30 split，没有 validation set。这意味着你无法做 early stopping，无法判断模型是欠拟合还是过拟合，也无法做超参数搜索。
MASTER 的标准做法是使用 Qlib 框架，训练集到 2020 Q1，验证集是 2020 Q2，测试集是 2020 Q3 到 2022 Q4 GitHub。更严格的做法是 rolling window——每隔一段时间用最近 N 年数据重新训练模型，模拟真实交易中的定期更新。
你的模型用 2016-2022 的数据训练，然后去预测 2023-2025——但市场结构在这三年里发生了巨大变化（利率环境、AI 热潮、地缘风险），一个 2022 年之前训练的模型很可能已经"过期"了。

缺陷 4（Moderate）：Loss Function 的排序信号太弱
你的 rank loss 是随机采样 pair 对比较，这种方式效率很低。最新研究在评估各种 ranking loss 对 Transformer 股票选择能力的影响 Springer，发现 listwise 方法（如 ListMLE、ListNet）比 pairwise 方法更有效，因为它直接优化整个排名序列，而不是一对一对比较。
另外，MASTER 使用 CSZcoreNorm 对标签做横截面标准化——按每天分组，用当天所有股票的 mean/std 归一化 target GitHub。这消除了不同市场状态下绝对收益率的差异，让模型专注学习"相对排名"。

缺陷 5（Moderate）：不具备市场状态适应能力
MASTER 的一个核心创新是利用市场指数信息来指导特征选择——在牛市和熊市中，有效的因子是不同的 arXiv。你的模型虽然输入了与 SP500 的相关性因子，但没有让市场状态动态影响模型内部的计算方式。
最近的研究也表明，在低信噪比环境下，对 attention 矩阵做动态稀疏化能带来 10-20% 的性能提升 arXiv，因为金融数据中大部分 attention 权重是噪声。

<img src="./update_img/JP_Morgen_V4_Update.png">

## 0. 依赖

In [1]:
!pip install yfinance -q

In [2]:
# =====================================================================
# Cell 0.5: 日志系统（在 import cell 之后、Cell 1 之前插入）
# =====================================================================
# 目录结构:
#   ./log/V4_2026-04-01_1646/
#     ├── log.out      ← 所有 print 输出
#     ├── log.err      ← 错误信息
#     └── img/         ← 图片
#         ├── 01_xxx.png
#         └── 02_xxx.png

import sys
import io
from datetime import datetime
import os
from matplotlib import pyplot as plt

MODEL_VERSION = "V4"
LOG_ROOT = "/Users/yigewang/Desktop/Quant_AI/JP_Morgen/log"

# 生成运行目录
timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
RUN_NAME = f"{MODEL_VERSION}_{timestamp}"
RUN_DIR = os.path.join(LOG_ROOT, RUN_NAME)
LOG_FILE = os.path.join(RUN_DIR, "log.out")
ERR_FILE = os.path.join(RUN_DIR, "log.err")
IMG_DIR = os.path.join(RUN_DIR, "img")

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)


class TeeWriter:
    """同时写入 notebook cell 和文件"""
    def __init__(self, file_path, original_stream):
        self.file = open(file_path, 'a', buffering=1)
        self.original = original_stream

    def write(self, msg):
        self.original.write(msg)
        self.file.write(msg)
        self.file.flush()

    def flush(self):
        self.original.flush()
        self.file.flush()

    def close(self):
        self.file.close()


# 保存当前的 stdout/stderr（Jupyter 的 IO 流）
_notebook_stdout = sys.stdout
_notebook_stderr = sys.stderr

# 重定向：同时写入 notebook cell + 文件
sys.stdout = TeeWriter(LOG_FILE, _notebook_stdout)
sys.stderr = TeeWriter(ERR_FILE, _notebook_stderr)


# 重写 plt.show() 让图片自动保存
_original_plt_show = plt.show
_img_counter = [0]

def auto_save_show(*args, **kwargs):
    _img_counter[0] += 1
    fig = plt.gcf()
    title = fig._suptitle.get_text() if fig._suptitle else f"figure_{_img_counter[0]:02d}"
    safe_title = "".join(c if c.isalnum() or c in (' ', '-', '_') else '_' for c in title)
    safe_title = safe_title.strip().replace(' ', '_')[:80] or f"figure_{_img_counter[0]:02d}"
    save_path = os.path.join(IMG_DIR, f"{_img_counter[0]:02d}_{safe_title}.png")
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"[LOG] Saved: {save_path}")
    _original_plt_show(*args, **kwargs)

plt.show = auto_save_show


print(f"{'='*60}")
print(f"  Run:  {RUN_NAME}")
print(f"  Dir:  {RUN_DIR}/")
print(f"  Out:  log.out")
print(f"  Err:  log.err")
print(f"  Img:  img/")
print(f"{'='*60}\n")

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import yfinance as yf
import matplotlib.pyplot as plt
from scipy import stats
import os, pickle, copy
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. 下载数据（首次运行）

In [4]:

# =====================================================================
# Cell 1 中替换 STOCK_UNIVERSE（~280 只股票）
# =====================================================================
# 按行业分组，方便后续做行业中性化

STOCK_UNIVERSE = {
    # ── Mega Cap Tech (15) ──
    'mega_tech': [
        'AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA','AVGO',
        'ORCL','CRM','ADBE','AMD','INTC','CSCO','IBM',
    ],
    # ── Semiconductors (15) ──
    'semicon': [
        'QCOM','TXN','MU','AMAT','LRCX','KLAC','MRVL','ON',
        'ADI','SNPS','CDNS','NXPI','MCHP','SWKS','TER',
    ],
    # ── Cloud / SaaS / Cybersecurity (20) ──
    'cloud_saas': [
        'NOW','PANW','CRWD','NET','DDOG','ZS','SNOW','PLTR',
        'FTNT','ANET','INTU','WDAY','TEAM','HUBS','VEEV',
        'OKTA','MDB','BILL','ZI','ESTC',
    ],
    # ── Consumer Internet / Platforms (12) ──
    'consumer_internet': [
        'NFLX','PYPL','SQ','SHOP','UBER','ABNB','SNAP','PINS',
        'DASH','HOOD','APP','RBLX',
    ],
    # ── Energy — Oil & Gas (15) ──
    'energy_oil': [
        'XOM','CVX','COP','EOG','SLB','MPC','PSX','VLO',
        'OXY','DVN','HES','HAL','BKR','FANG','KMI',
    ],
    # ── Energy — Clean / Utilities (10) ──
    'energy_clean': [
        'FSLR','NEE','AES','CEG','VST','NRG','GEV',
        'ENPH','DUK','SO',
    ],
    # ── Financials — Banks & Asset Mgmt (20) ──
    'financials': [
        'JPM','GS','MS','BAC','WFC','C','SCHW','USB',
        'PNC','BK','COF','AXP','BLK','BX','KKR',
        'APO','ICE','CME','MCO','SPGI',
    ],
    # ── Insurance (6) ──
    'insurance': [
        'CB','PGR','TRV','AON','ALL','MET',
    ],
    # ── Healthcare — Pharma & Biotech (20) ──
    'pharma_biotech': [
        'LLY','JNJ','UNH','ABBV','MRK','PFE','AMGN','GILD',
        'BMY','REGN','VRTX','BIIB','MRNA','ILMN','SGEN',
        'ALNY','INCY','BGNE','EXAS','PCVX',
    ],
    # ── Healthcare — Devices & Services (12) ──
    'health_devices': [
        'ABT','TMO','ISRG','SYK','BSX','MDT','DHR','HCA',
        'EW','DXCM','PODD','HOLX',
    ],
    # ── Emerging Biotech / MedTech (10) ──
    'emerging_biotech': [
        'CRSP','NTLA','BEAM','EDIT','RARE','TWST',
        'RXRX','SDGR','TXG','NUVB',
    ],
    # ── Industrials / Aerospace (15) ──
    'industrials': [
        'CAT','GE','RTX','HON','BA','LMT','GD','NOC',
        'DE','ETN','PH','TT','LHX','TDG','HWM',
    ],
    # ── Consumer Staples (12) ──
    'consumer_staples': [
        'PG','KO','PEP','COST','WMT','MCD','SBUX','CL',
        'MDLZ','MNST','PM','MO',
    ],
    # ── Consumer Discretionary (12) ──
    'consumer_disc': [
        'HD','LOW','TJX','ROST','NKE','MAR','HLT','RCL',
        'GM','ORLY','BKNG','DIS',
    ],
    # ── Telecom / Media (6) ──
    'telecom': [
        'TMUS','VZ','T','CMCSA','WBD','CHTR',
    ],
    # ── REITs / Infrastructure (8) ──
    'reits': [
        'PLD','AMT','EQIX','DLR','SPG','WELL','CCI','PSA',
    ],
    # ── Materials / Mining (6) ──
    'materials': [
        'LIN','SHW','ECL','APD','NEM','FCX',
    ],
    # ── Transport / Logistics (6) ──
    'transport': [
        'UNP','CSX','NSC','FDX','UPS','WM',
    ],
    # ── Emerging AI / Robotics (8) ──
    'emerging_ai': [
        'PATH','AI','IONQ','SMCI','SOUN','BBAI','GFAI','AMBA',
    ],
}

In [5]:
# 展平成列表
STOCK_UNIVERSE_FLAT = []
STOCK_SECTOR_MAP = {}  # ticker -> sector（后续行业中性化用）
for sector, tickers in STOCK_UNIVERSE.items():
    for t in tickers:
        if t not in STOCK_UNIVERSE_FLAT:
            STOCK_UNIVERSE_FLAT.append(t)
            STOCK_SECTOR_MAP[t] = sector

print(f"Stock universe: {len(STOCK_UNIVERSE_FLAT)} tickers across {len(STOCK_UNIVERSE)} sectors")
for sector, tickers in STOCK_UNIVERSE.items():
    print(f"  {sector:25s}: {len(tickers)} stocks")

In [6]:
## 1. 下载数据（带日期范围校验）

MARKET_TICKERS = {'SPY':'SP500','QQQ':'Nasdaq100','XLE':'EnergySector','TLT':'Bond20Y','GLD':'Gold'}

# ========== 你只需要改这里 ==========
START_DATE = '2016-01-01'
END_DATE   = '2025-12-31'
# =====================================

SAVE_DIR = './v3_data'
META_FILE = os.path.join(SAVE_DIR, 'meta.json')  # 新增：记录数据的日期范围

import json

def save_meta(start, end, num_stocks):
    """保存数据的元信息（日期范围、股票数量）"""
    meta = {
        'start_date': start,
        'end_date': end,
        'num_stocks': num_stocks,
        'saved_at': pd.Timestamp.now().isoformat()
    }
    with open(META_FILE, 'w') as f:
        json.dump(meta, f, indent=2)

def check_date_range(start, end):
    """
    检查本地数据的日期范围是否覆盖请求的范围
    返回: (is_valid, reason)
    """
    if not os.path.exists(META_FILE):
        return False, '未找到 meta.json（旧版数据，无日期记录）'
    
    with open(META_FILE, 'r') as f:
        meta = json.load(f)
    
    saved_start = meta.get('start_date', '')
    saved_end   = meta.get('end_date', '')
    
    # 检查: 本地数据的起始日期 <= 请求的起始日期
    #       本地数据的结束日期 >= 请求的结束日期
    if saved_start > start:
        return False, f'本地数据起始 {saved_start} 晚于请求的 {start}'
    if saved_end < end:
        return False, f'本地数据结束 {saved_end} 早于请求的 {end}'
    
    print(f'✅ 日期校验通过: 本地 [{saved_start} ~ {saved_end}] 覆盖请求 [{start} ~ {end}]')
    return True, 'OK'

def download_all(start=START_DATE, end=END_DATE):
    print(f'下载股票数据 [{start} ~ {end}]...')
    raw = {}
    for t in STOCK_UNIVERSE_FLAT:
        try:
            df = yf.download(t, start=start, end=end, progress=False)
            if len(df) > 200:
                raw[t] = df
                print(f'  ✓ {t}: {len(df)}')
        except: print(f'  ✗ {t}')
    
    print('\n下载市场基准...')
    mkt = {}
    for t, name in MARKET_TICKERS.items():
        try:
            df = yf.download(t, start=start, end=end, progress=False)
            if len(df) > 200:
                mkt[name] = df['Close'].values.flatten()
                mkt[f'{name}_idx'] = df.index
                print(f'  ✓ {name}')
        except: print(f'  ✗ {name}')
    
    print('\n获取基本面...')
    fund = {}
    for t in raw.keys():
        try:
            info = yf.Ticker(t).info
            fund[t] = {
                'pe': info.get('trailingPE', np.nan),
                'pb': info.get('priceToBook', np.nan),
                'market_cap': info.get('marketCap', np.nan),
                'dividend_yield': info.get('dividendYield', 0) or 0,
                'roe': info.get('returnOnEquity', np.nan),
            }
            print(f'  ✓ {t}')
        except:
            fund[t] = {}
            print(f'  ✗ {t}')
    
    return raw, mkt, fund

def save_data(raw, mkt, fund, start=START_DATE, end=END_DATE):
    os.makedirs(SAVE_DIR, exist_ok=True)
    for t, df in raw.items():
        df_save = df.copy()
        df_save.columns = [str(c).split(',')[0].strip().strip("('\"") for c in df_save.columns]
        df_save.to_csv(os.path.join(SAVE_DIR, f'{t}.csv'))
    with open(os.path.join(SAVE_DIR, 'market.pkl'), 'wb') as f:
        pickle.dump(mkt, f)
    with open(os.path.join(SAVE_DIR, 'fund.pkl'), 'wb') as f:
        pickle.dump(fund, f)
    
    # 新增：保存元信息
    save_meta(start, end, len(raw))
    print(f'✅ 保存到 {SAVE_DIR}/ ({len(raw)} stocks, {start} ~ {end})')

def load_data(start=START_DATE, end=END_DATE):
    raw = {}
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    for fname in os.listdir(SAVE_DIR):
        if fname.endswith('.csv'):
            t = fname.replace('.csv', '')
            df = pd.read_csv(os.path.join(SAVE_DIR, fname), index_col=0, parse_dates=True)
            # 清洗列名 + 强制数值类型
            df.columns = [str(col).split(',')[0].strip().strip("('\"") for col in df.columns]
            for col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            df = df.dropna(how='all')
            # 日期过滤
            df = df[(df.index >= start_ts) & (df.index <= end_ts)]
            if len(df) > 200:
                raw[t] = df
    with open(os.path.join(SAVE_DIR, 'market.pkl'), 'rb') as f:
        mkt = pickle.load(f)
    with open(os.path.join(SAVE_DIR, 'fund.pkl'), 'rb') as f:
        fund = pickle.load(f)
    for name in ['SP500','Nasdaq100','EnergySector','Bond20Y','Gold']:
        if name in mkt and f'{name}_idx' in mkt:
            idx = mkt[f'{name}_idx']
            mask = (idx >= start_ts) & (idx <= end_ts)
            mkt[name] = mkt[name][mask]
            mkt[f'{name}_idx'] = idx[mask]
    print(f'✅ 从 {SAVE_DIR}/ 加载 ({len(raw)} stocks, filtered to [{start} ~ {end}])')
    return raw, mkt, fund

# ========== 主逻辑：校验 → 加载 or 重新下载 ==========

data_valid = False

if os.path.exists(os.path.join(SAVE_DIR, 'market.pkl')):
    data_valid, reason = check_date_range(START_DATE, END_DATE)
    if not data_valid:
        print(f'⚠️  日期不匹配: {reason}')
        print(f'   正在重新下载 [{START_DATE} ~ {END_DATE}]...\n')

if data_valid:
    raw_data, market_data, fundamentals = load_data()
else:
    raw_data, market_data, fundamentals = download_all(START_DATE, END_DATE)
    save_data(raw_data, market_data, fundamentals, START_DATE, END_DATE)

## 2. V3 因子工程（~30 factors）

In [7]:
def compute_v3_factors(df, market_data, fund_data=None):
    f = pd.DataFrame(index=df.index)
    c = pd.Series(df['Close'].values.flatten(), index=df.index)
    v = pd.Series(df['Volume'].values.flatten(), index=df.index)
    h = pd.Series(df['High'].values.flatten(), index=df.index)
    l = pd.Series(df['Low'].values.flatten(), index=df.index)
    o = pd.Series(df['Open'].values.flatten(), index=df.index)
    dr = c.pct_change()

    for n in [5,10,20,60]: f[f'ret_{n}d'] = c.pct_change(n)
    f['mom_accel'] = f['ret_5d'] - f['ret_5d'].shift(5)

    f['vol_20d'] = dr.rolling(20).std()
    f['vol_ratio'] = dr.rolling(5).std() / dr.rolling(20).std()
    f['vol_accel'] = f['vol_20d'].pct_change(5)
    f['intraday_vol'] = ((h - l) / c).rolling(20).mean()
    f['vol_spread'] = f['intraday_vol'] - f['vol_20d']

    f['vol_ma_ratio'] = v / v.rolling(20).mean()
    f['volume_trend'] = v.rolling(5).mean() / v.rolling(20).mean() - 1
    obv = (dr.apply(np.sign) * v).rolling(20).sum()
    f['obv_norm'] = obv / v.rolling(20).sum()
    vwap = (c * v).rolling(20).sum() / v.rolling(20).sum()
    f['vwap_dev'] = c / vwap - 1

    f['bb_pos'] = (c - c.rolling(20).mean()) / (c.rolling(20).std() * 2)
    f['ma_cross_s'] = c.rolling(5).mean() / c.rolling(20).mean() - 1
    f['ma_cross_l'] = c.rolling(20).mean() / c.rolling(60).mean() - 1

    delta = c.diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss_v = (-delta.where(delta < 0, 0)).rolling(14).mean()
    f['rsi'] = (100 - 100 / (1 + gain / loss_v.replace(0, np.nan))) / 100 - 0.5

    rmin, rmax = c.rolling(60).min(), c.rolling(60).max()
    f['price_pos'] = ((c - rmin) / (rmax - rmin).replace(0, np.nan)) - 0.5
    f['price_range'] = (h - l) / c

    for name in ['SP500','Nasdaq100','EnergySector']:
        if name in market_data:
            ref = pd.Series(market_data[name], index=market_data[f'{name}_idx'])
            ref_a = ref.reindex(df.index, method='ffill')
            f[f'rel_{name}'] = c.pct_change(20) - ref_a.pct_change(20)
            f[f'corr_{name}'] = dr.rolling(60).corr(ref_a.pct_change())

    f['ret_252d'] = c.pct_change(252)
    f['ma200_dev'] = c / c.rolling(200).mean() - 1
    f['sharpe_20d'] = f['ret_20d'] / f['vol_20d'].replace(0, np.nan)
    f['gap'] = (o / c.shift(1) - 1).rolling(5).mean()

    if fund_data:
        for k in ['pe','pb','market_cap','dividend_yield','roe']:
            val = fund_data.get(k, np.nan)
            f[f'f_{k}'] = val if (not np.isnan(val) if isinstance(val, float) else True) else 0.0

    return f.dropna()

sample = compute_v3_factors(raw_data[list(raw_data.keys())[0]], market_data, fundamentals.get(list(raw_data.keys())[0], {}))
print(f'因子数: {sample.shape[1]}')
print(list(sample.columns))

## 3. 数据集（滚动标准化 + 日期对齐）

## update 1: tranfer to rolling window

<img src="./update_img/JP_Morgen_V4_rolling_window.png">

In [8]:

# =====================================================================
# Cell 3a: 配置参数
# =====================================================================

LOOKBACK = 20
SCALER_WINDOW = 120
MIN_HISTORY_YEARS = 5
MIN_STOCKS_RATIO = 0.80   # 每天至少 80% 股票有数据
MIN_STOCKS_PER_DAY = 20   # 每天至少 20 只股票才组成样本

In [9]:
#  =====================================================================
# Cell 3b: 计算因子 + 过滤短历史股票
# =====================================================================

all_stock = {}
for t, df in raw_data.items():
    fac = compute_v3_factors(df, market_data, fundamentals.get(t, {}))
    cs = pd.Series(df['Close'].values.flatten(), index=df.index)
    fut = cs.pct_change(5).shift(-5)
    idx = fac.index.intersection(fut.dropna().index)
    if len(idx) == 0:
        continue
    all_stock[t] = {'factors': fac.loc[idx], 'target': fut.loc[idx]}

# 过滤历史太短的股票
cutoff = pd.Timestamp.now() - pd.DateOffset(years=MIN_HISTORY_YEARS)
short = [t for t in all_stock if all_stock[t]['factors'].index.min() > cutoff]
if short:
    print(f"⚠️  Removing {len(short)} stocks with < {MIN_HISTORY_YEARS} years:")
    for t in short:
        print(f"    {t}: starts {all_stock[t]['factors'].index.min().strftime('%Y-%m-%d')}")
        del all_stock[t]

NUM_FACTORS = list(all_stock.values())[0]['factors'].shape[1]
print(f"\n✅ Keeping {len(all_stock)} stocks | {NUM_FACTORS} factors")

In [10]:
# =====================================================================
# Cell 3c: 计算有效交易日（宽松交集）
# =====================================================================

from collections import Counter

date_counts = Counter()
for d in all_stock.values():
    for dt in d['factors'].index:
        date_counts[dt] += 1

n_stocks = len(all_stock)
min_per_day = int(n_stocks * MIN_STOCKS_RATIO)
all_dates = sorted([dt for dt, cnt in date_counts.items() if cnt >= min_per_day])

print(f"Require >= {min_per_day}/{n_stocks} stocks per day ({MIN_STOCKS_RATIO:.0%})")
print(f"Valid trading days: {len(all_dates)}")
print(f"Date range: {all_dates[0].strftime('%Y-%m-%d')} ~ {all_dates[-1].strftime('%Y-%m-%d')}")



In [11]:
# =====================================================================
# Cell 3d: 滚动标准化 + 按天组织数据
# =====================================================================

all_dates_set = set(all_dates)

# 为每只股票预计算标准化后的因子窗口
stock_data = {}  # {ticker: {date: {'x': array, 'y': float}}}

for t, d in all_stock.items():
    fdf = d['factors']
    tgt = d['target']
    fn = fdf.values
    dates_idx = fdf.index

    norm_rows = {}
    for i in range(max(SCALER_WINDOW, LOOKBACK), len(fn)):
        w = fn[max(0, i - SCALER_WINDOW):i]
        mu, sd = np.nanmean(w, axis=0), np.nanstd(w, axis=0)
        sd[sd < 1e-8] = 1.0
        s = (fn[i - LOOKBACK:i] - mu) / sd

        if s.shape != (LOOKBACK, NUM_FACTORS) or np.any(np.isnan(s)):
            continue
        dt = dates_idx[i]
        if dt in all_dates_set and not np.isnan(tgt.iloc[i]):
            norm_rows[dt] = {'x': s.astype(np.float32), 'y': tgt.iloc[i]}

    if len(norm_rows) > 0:
        stock_data[t] = norm_rows

tickers_list = sorted(stock_data.keys())
print(f"Stocks with valid normalized data: {len(tickers_list)}")

# 按天组织
daily_samples = {}  # {date: {'X': (N,T,F), 'y': (N,), 'tickers': [...]}}

for dt in all_dates:
    day_x, day_y, day_tickers = [], [], []
    for t in tickers_list:
        if dt in stock_data[t]:
            day_x.append(stock_data[t][dt]['x'])
            day_y.append(stock_data[t][dt]['y'])
            day_tickers.append(t)
    if len(day_x) >= MIN_STOCKS_PER_DAY:
        daily_samples[dt] = {
            'X': np.stack(day_x),
            'y': np.array(day_y, dtype=np.float32),
            'tickers': day_tickers
        }

valid_dates = sorted(daily_samples.keys())
n_per_day = [len(daily_samples[d]['tickers']) for d in valid_dates]
print(f"\nDaily dataset: {len(valid_dates)} days")
print(f"Stocks/day: min={min(n_per_day)}, max={max(n_per_day)}, avg={np.mean(n_per_day):.0f}")
print(f"Date range: {valid_dates[0].strftime('%Y-%m-%d')} ~ {valid_dates[-1].strftime('%Y-%m-%d')}")



In [12]:
# ====================================================================
# Cell 3e: 打包 full_data + DailyDataset 类
# =====================================================================

full_data = {
    'daily_samples': daily_samples,
    'valid_dates': valid_dates,
    'tickers_list': tickers_list,
    'num_factors': NUM_FACTORS,
}

class DailyDataset(Dataset):
    """每个 __getitem__ 返回一天的全部股票"""
    def __init__(self, dates, daily_samples, max_stocks=None):
        self.dates = dates
        self.daily_samples = daily_samples
        self.max_stocks = max_stocks or max(
            len(daily_samples[dt]['tickers']) for dt in dates
        )

    def __len__(self):
        return len(self.dates)

    def __getitem__(self, idx):
        dt = self.dates[idx]
        sample = self.daily_samples[dt]
        X = torch.FloatTensor(sample['X'])      # (N, T, F)
        y = torch.FloatTensor(sample['y'])      # (N,)
        N = X.shape[0]

        # Pad 到 max_stocks
        if N < self.max_stocks:
            pad_x = torch.zeros(self.max_stocks - N, X.shape[1], X.shape[2])
            pad_y = torch.zeros(self.max_stocks - N)
            X = torch.cat([X, pad_x], dim=0)
            y = torch.cat([y, pad_y], dim=0)

        mask = torch.zeros(self.max_stocks)
        mask[:N] = 1.0

        return X, y, mask

print(f"\n✅ full_data ready | {len(valid_dates)} days | {NUM_FACTORS} factors")
print(f"   Next: run Cell 4 (model) → Cell 5 (loss) → Cell 6 (train)")

In [13]:
class SDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

## 4. V3 Model

In [14]:

# =====================================================================
# Cell 4: V4 Model — Two-Way Attention
# =====================================================================

class Time2Vec(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.lin = nn.Linear(1, 1)
        self.per = nn.Linear(1, d - 1)
    def forward(self, T, dev):
        t = (torch.arange(T, dtype=torch.float32, device=dev) / T).unsqueeze(-1)
        return torch.cat([self.lin(t), torch.sin(self.per(t))], dim=-1)


class TwoWayBlock(nn.Module):
    """
    交替 Temporal + Cross-Sectional Attention
    
    数据流:
      输入 x: (N_stocks, T, d)
      Step 1 - Temporal:  每只股票独立看自己的历史 → (N, T, d)
      Step 2 - Cross-Sec: 转置 → (T, N, d)，同一天的股票互相看
      Step 3 - 转回:      (N, T, d)
    """
    def __init__(self, d, nh, ff, drop):
        super().__init__()
        self.temporal = nn.TransformerEncoderLayer(
            d_model=d, nhead=nh, dim_feedforward=ff,
            dropout=drop, batch_first=True, norm_first=True
        )
        self.cross_sec = nn.TransformerEncoderLayer(
            d_model=d, nhead=nh, dim_feedforward=ff,
            dropout=drop, batch_first=True, norm_first=True
        )

    def forward(self, x, stock_mask=None):
        # x: (N, T, d), stock_mask: (N,) — 1=real, 0=pad

        # Step 1: Temporal attention (每只股票独立)
        x = self.temporal(x)  # (N, T, d)

        # Step 2: Cross-sectional attention (同一天的股票互相看)
        T = x.shape[1]
        x = x.permute(1, 0, 2)  # (T, N, d)

        # 用 mask 防止 attention 看到 padding 的股票
        if stock_mask is not None:
            # key_padding_mask: True = 忽略该位置
            pad_mask = (stock_mask == 0).unsqueeze(0).expand(T, -1)  # (N,T)
        else:
            pad_mask = None

        x = self.cross_sec(x, src_key_padding_mask=pad_mask)  # (T, N, d)
        x = x.permute(1, 0, 2)  # (N, T, d)

        return x


class QuantV4(nn.Module):
    """
    V4 架构:
    Input: (N_stocks, T, F) — 一天的全部股票
    → Linear Projection + Time2Vec
    → TwoWayBlock × N layers (temporal + cross-sectional attention)
    → Attention Pooling over time
    → Prediction head
    Output: (N_stocks, 1) — 每只股票的预测值
    """
    def __init__(self, nf, d=128, nh=4, nl=3, ff=256, drop=0.15):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(nf, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(drop)
        )
        self.t2v = Time2Vec(d)
        self.blocks = nn.ModuleList([
            TwoWayBlock(d, nh, ff, drop) for _ in range(nl)
        ])
        # Attention pooling over time dimension
        self.pool = nn.Sequential(nn.Linear(d, d // 4), nn.Tanh(), nn.Linear(d // 4, 1))
        # Prediction head
        self.head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d // 2), nn.GELU(), nn.Dropout(drop),
            nn.Linear(d // 2, 1)
        )
        self.apply(self._iw)

    def _iw(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x, stock_mask=None):
        # x: (N, T, F)
        N, T, _ = x.shape
        x = self.proj(x) + self.t2v(T, x.device).unsqueeze(0)  # (N, T, d)

        for block in self.blocks:
            x = block(x, stock_mask)  # (N, T, d)

        # Attention pooling: 把 T 个时间步压成 1 个
        w = torch.softmax(self.pool(x), dim=1)  # (N, T, 1)
        x = (x * w).sum(dim=1)  # (N, d)

        return self.head(x).squeeze(-1)  # (N,)


# 初始化
model = QuantV4(nf=NUM_FACTORS, d=128, nh=4, nl=3, ff=256, drop=0.15).to(device)
print(f'V4 params: {sum(p.numel() for p in model.parameters()):,}')

## 5. Combined Loss (Rank + MSE + IC)

In [15]:
#  =====================================================================
# Cell 5: ListNet Loss + Combined Loss
# =====================================================================

class ListNetLoss(nn.Module):
    """
    ListNet: softmax 交叉熵 — 直接优化整个排名列表
    要求输入是同一天的全部股票（不是随机 batch）
    """
    def __init__(self, temperature=1.0):
        super().__init__()
        self.temp = temperature

    def forward(self, pred, target, mask=None):
        """
        pred:   (N,) 模型预测
        target: (N,) 真实标签
        mask:   (N,) 1=real, 0=padding
        """
        if mask is not None:
            # 只看真实股票
            pred = pred[mask > 0]
            target = target[mask > 0]

        if len(pred) < 5:
            return torch.tensor(0.0, device=pred.device)

        p_true = torch.softmax(target / self.temp, dim=0)
        p_pred = torch.log_softmax(pred / self.temp, dim=0)
        return -torch.sum(p_true * p_pred)


class V4CombinedLoss(nn.Module):
    """
    V4 Loss = ListNet + IC regularization
    - ListNet: 直接优化排名（listwise）
    - IC: 额外的 correlation 信号
    """
    def __init__(self, listnet_weight=0.7, ic_weight=0.3, temperature=1.0):
        super().__init__()
        self.lw = listnet_weight
        self.iw = ic_weight
        self.listnet = ListNetLoss(temperature=temperature)

    def forward(self, pred, target, mask=None):
        if mask is not None:
            p = pred[mask > 0]
            t = target[mask > 0]
        else:
            p, t = pred, target

        if len(p) < 5:
            return torch.tensor(0.0, device=pred.device), 0.0, 0.0

        # ListNet loss
        ln_loss = self.listnet(pred, target, mask)

        # IC regularization (maximize correlation)
        pm, tm = p - p.mean(), t - t.mean()
        ic = (pm * tm).mean() / ((pm.std() + 1e-8) * (tm.std() + 1e-8))

        total = self.lw * ln_loss + self.iw * (1 - ic)
        return total, ln_loss.item(), ic.item()


criterion = V4CombinedLoss(listnet_weight=0.7, ic_weight=0.3)
print("Loss: 70% ListNet + 30% IC")

## 6. 训练（CosineAnnealingLR T_max=40, 80 epochs）

In [16]:

# =====================================================================
# Cell 6: V4 Rolling Window 训练
# =====================================================================

from dateutil.relativedelta import relativedelta


def generate_folds(dates, val_months=6, test_months=6, min_train_years=3):
    unique_dates = sorted(set([pd.Timestamp(d).to_pydatetime() for d in dates]))
    first_date, last_date = unique_dates[0], unique_dates[-1]

    folds = []
    test_start = first_date + relativedelta(years=min_train_years, months=val_months)

    while True:
        test_end = test_start + relativedelta(months=test_months)
        if test_end > last_date:
            if test_start < last_date:
                test_end = last_date
            else:
                break

        val_start = test_start - relativedelta(months=val_months)
        train_end = val_start

        tr = [d for d in unique_dates if first_date <= d < train_end]
        va = [d for d in unique_dates if val_start <= d < test_start]
        te = [d for d in unique_dates if test_start <= d < test_end]

        if len(tr) > 100 and len(va) > 20 and len(te) > 20:
            folds.append({
                'train_start': first_date, 'train_end': train_end,
                'val_start': val_start, 'val_end': test_start,
                'test_start': test_start, 'test_end': test_end,
                'n_train': len(tr), 'n_val': len(va), 'n_test': len(te),
            })

        test_start = test_end
        if test_end >= last_date:
            break

    return folds


def get_dates_in_range(all_dates, start, end):
    """返回 [start, end) 范围内的日期列表"""
    s = pd.Timestamp(start).to_pydatetime()
    e = pd.Timestamp(end).to_pydatetime()
    return [d for d in all_dates if s <= pd.Timestamp(d).to_pydatetime() < e]


def train_one_fold_v4(fold, full_data, num_epochs=60, patience=8, lr=3e-4):
    """V4 训练：按天 DataLoader + Two-Way Attention"""

    daily = full_data['daily_samples']
    all_dates = full_data['valid_dates']

    tr_dates = get_dates_in_range(all_dates, fold['train_start'], fold['train_end'])
    va_dates = get_dates_in_range(all_dates, fold['val_start'], fold['val_end'])
    te_dates = get_dates_in_range(all_dates, fold['test_start'], fold['test_end'])

    # 只保留在 daily_samples 中存在的日期
    tr_dates = [d for d in tr_dates if d in daily]
    va_dates = [d for d in va_dates if d in daily]
    te_dates = [d for d in te_dates if d in daily]

    print(f"  Train: {len(tr_dates)} days | Val: {len(va_dates)} days | Test: {len(te_dates)} days")

    if len(tr_dates) < 50 or len(va_dates) < 10 or len(te_dates) < 10:
        print("  ⚠️ Not enough data, skipping")
        return pd.DataFrame(), 0.0, 0.0

    # ---- DataLoader ----
    # 找 max_stocks 用于 padding（跨 train/val/test）
    max_stocks = max(
        max(len(daily[d]['tickers']) for d in tr_dates),
        max(len(daily[d]['tickers']) for d in va_dates),
        max(len(daily[d]['tickers']) for d in te_dates),
    )

    # train_ds = DailyDataset(tr_dates, daily, max_stocks=max_stocks)
    # # batch_size=1 因为每天已经是一个完整的"batch"（所有股票）
    # # shuffle=True 只打乱天的顺序，同一天内的股票保持完整
    # train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
    #                       num_workers=4, pin_memory=True, persistent_workers=True)
    # ---- 原来的代码（删掉）----
    # train_ds = DailyDataset(tr_dates, daily, max_stocks=max_stocks)
    # train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)

    # ---- 替换成预计算版本 ----
    precomputed = {}
    for dt in tr_dates:
        sample = daily[dt]
        X = torch.FloatTensor(sample['X'])
        y = torch.FloatTensor(sample['y'])
        N = X.shape[0]
        if N < max_stocks:
            X = torch.cat([X, torch.zeros(max_stocks - N, X.shape[1], X.shape[2])])
            y = torch.cat([y, torch.zeros(max_stocks - N)])
        mask = torch.zeros(max_stocks)
        mask[:N] = 1.0
        precomputed[dt] = (X.to(device), y.to(device), mask.to(device))
    
    print(f"  Precomputed {len(precomputed)} days")

    # ---- Model ----
    nf = full_data['num_factors']
    model = QuantV4(nf=nf, d=128, nh=4, nl=3, ff=256, drop=0.15).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs // 2)
    criterion = V4CombinedLoss(listnet_weight=0.7, ic_weight=0.3)

    best_val_ic = -np.inf
    best_state = None
    no_improve = 0

    for ep in range(num_epochs):
        # ---- Train ----
        model.train()
        epoch_loss, epoch_ic, n_batch = 0, 0, 0
        
        # 打乱天的顺序
        shuffled_dates = tr_dates.copy()
        np.random.shuffle(shuffled_dates)
        
        for dt in shuffled_dates:
            X, y, mask = precomputed[dt]

            optimizer.zero_grad()
            pred = model(X, stock_mask=mask)
            loss, ln, ic = criterion(pred, y, mask)

            if loss.requires_grad:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            epoch_loss += loss.item()
            epoch_ic += ic
            n_batch += 1

        scheduler.step()

        # ---- Validate ----
        model.eval()
        val_ics = []
        with torch.no_grad():
            for dt in va_dates:
                sample = daily[dt]
                X_v = torch.FloatTensor(sample['X']).to(device)
                y_v = sample['y']
                pred_v = model(X_v).cpu().numpy()
                ic, _ = stats.spearmanr(pred_v, y_v)
                if not np.isnan(ic):
                    val_ics.append(ic)

        val_ic = np.mean(val_ics) if val_ics else 0.0

        if val_ic > best_val_ic:
            best_val_ic = val_ic
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if (ep + 1) % 10 == 0:
            print(f"    Ep {ep+1:3d} | Loss:{epoch_loss/max(n_batch,1):.4f} | "
                  f"Train IC:{epoch_ic/max(n_batch,1):.4f} | Val IC:{val_ic:.4f} | "
                  f"Best:{best_val_ic:.4f} | P:{no_improve}/{patience}")

        if no_improve >= patience:
            print(f"    Early stop at epoch {ep+1} (best Val IC: {best_val_ic:.4f})")
            break

    # ---- Test ----
    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    results = []
    with torch.no_grad():
        for dt in te_dates:
            sample = daily[dt]
            X_t = torch.FloatTensor(sample['X']).to(device)
            pred_t = model(X_t).cpu().numpy()
            for i, t in enumerate(sample['tickers']):
                results.append({
                    'ticker': t,
                    'date': dt,
                    'predicted': pred_t[i],
                    'actual': sample['y'][i]  # raw return
                })

    result_df = pd.DataFrame(results)
    if len(result_df) > 0:
        test_ic, _ = stats.spearmanr(result_df['predicted'], result_df['actual'])
    else:
        test_ic = 0.0
    print(f"  ✅ Fold done | Test IC: {test_ic:.4f} | Best Val IC: {best_val_ic:.4f}\n")

    return result_df, best_val_ic, test_ic

In [ ]:
#  =====================================================================
# 主训练流程
# =====================================================================

valid_dates_dt = [pd.Timestamp(d).to_pydatetime() for d in full_data['valid_dates']]
date_range_years = (max(valid_dates_dt) - min(valid_dates_dt)).days / 365.25
print(f"Data spans {date_range_years:.1f} years")

if date_range_years >= 7:
    mt, vm, tm = 3, 6, 6
elif date_range_years >= 4:
    mt, vm, tm = 2, 4, 4
else:
    mt, vm, tm = 1, 3, 3
print(f"Auto config: min_train={mt}yr, val={vm}mo, test={tm}mo")

folds = generate_folds(valid_dates_dt, val_months=vm, test_months=tm, min_train_years=mt)

print(f"\n共 {len(folds)} 个 fold:\n")
for i, f in enumerate(folds):
    print(f"  Fold {i+1}: Train [{f['train_start'].strftime('%Y-%m')} ~ {f['train_end'].strftime('%Y-%m')}] "
          f"| Val [{f['val_start'].strftime('%Y-%m')} ~ {f['val_end'].strftime('%Y-%m')}] "
          f"| Test [{f['test_start'].strftime('%Y-%m')} ~ {f['test_end'].strftime('%Y-%m')}] "
          f"({f['n_train']}/{f['n_val']}/{f['n_test']} days)")

# ---- 训练 ----
all_results = []
fold_metrics = []

for i, fold in enumerate(folds):
    print(f"{'='*60}")
    print(f"Fold {i+1}/{len(folds)}: test [{fold['test_start'].strftime('%Y-%m-%d')} ~ {fold['test_end'].strftime('%Y-%m-%d')}]")
    print(f"{'='*60}")
    

    result, val_ic, test_ic = train_one_fold_v4(
        fold=fold,
        full_data=full_data,
        num_epochs=60,
        patience=8,
        lr=3e-4
    )

    if len(result) > 0:
        all_results.append(result)
        fold_metrics.append({
            'fold': i + 1,
            'test_start': fold['test_start'],
            'test_end': fold['test_end'],
            'val_ic': val_ic, 'test_ic': test_ic,
            'n_samples': len(result)
        })

# ---- Summary ----
if len(all_results) > 0:
    res_final = pd.concat(all_results, ignore_index=True).sort_values('date').reset_index(drop=True)
    metrics_df = pd.DataFrame(fold_metrics)

    print(f"\n{'='*60}")
    print("V4 Rolling Window Summary")
    print(f"{'='*60}")
    for _, row in metrics_df.iterrows():
        print(f"  Fold {row['fold']}: Val IC={row['val_ic']:.4f} | Test IC={row['test_ic']:.4f} | "
              f"[{row['test_start'].strftime('%Y-%m')} ~ {row['test_end'].strftime('%Y-%m')}]")

    print(f"\n  Avg Val IC:  {metrics_df['val_ic'].mean():.4f}")
    print(f"  Avg Test IC: {metrics_df['test_ic'].mean():.4f}")
    print(f"  IC gap: {metrics_df['val_ic'].mean() - metrics_df['test_ic'].mean():.4f}")
    print(f"\n  Total predictions: {len(res_final):,}")
else:
    print("❌ No results")

KeyboardInterrupt: 

In [ ]:
# =====================================================================
# Cell 6.5（新增）: Fold IC 可视化
# =====================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 每个 fold 的 val IC vs test IC
ax = axes[0]
x = range(1, len(fold_metrics) + 1)
ax.bar([i - 0.15 for i in x], metrics_df['val_ic'], width=0.3, color='steelblue', label='Val IC', alpha=0.8)
ax.bar([i + 0.15 for i in x], metrics_df['test_ic'], width=0.3, color='coral', label='Test IC', alpha=0.8)
ax.axhline(y=0, color='gray', ls='--', lw=0.5)
ax.axhline(y=0.05, color='green', ls='--', lw=0.5, label='IC=0.05 threshold')
ax.set_xlabel('Fold')
ax.set_ylabel('Spearman IC')
ax.set_title('Validation vs Test IC per Fold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 右图: 累计 IC 曲线（用合并后的 res_final）
ax = axes[1]
daily_ic = []
for dt, g in res_final.groupby('date'):
    if len(g) < 5:
        continue
    ic, _ = stats.spearmanr(g['predicted'], g['actual'])
    if not np.isnan(ic):
        daily_ic.append(ic)
ax.plot(np.cumsum(daily_ic), 'b-', lw=1.5)
ax.set_title(f'Cumulative IC (mean={np.mean(daily_ic):.4f})')
ax.set_xlabel('Trading days')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 回测（含交易成本 + 信号方向检测）

加入空仓判断，是否面对极度市场波动情况

In [ ]:
# =====================================================================
# Cell 7 替换：回测（直接用 res_final，不再需要 model 推理）
# =====================================================================

TX_COST = 0.001
REBAL_FREQ = 5
TOP_N = 3

def backtest_from_predictions(res, top_n=TOP_N, tx=TX_COST, rf=REBAL_FREQ, rev=False):
    """
    与原版的区别：
    - 输入是已有的预测 DataFrame（res_final），不再调用 model
    - 其余逻辑完全相同
    """
    res = res.copy()
    if rev:
        res['predicted'] = -res['predicted']

    dg = res.groupby('date')
    rd = sorted(res['date'].unique())[::rf]
    rows, pl, ps = [], set(), set()

    for dt in rd:
        if dt not in dg.groups:
            continue
        g = dg.get_group(dt)
        if len(g) < 2 * top_n:
            continue
        g = g.sort_values('predicted', ascending=False)
        lt = set(g.head(top_n)['ticker'])
        st = set(g.tail(top_n)['ticker'])
        lr = g.head(top_n)['actual'].mean()
        sr = -g.tail(top_n)['actual'].mean()
        to = (len(lt - pl) + len(st - ps)) / (2 * top_n)
        cost = to * tx * 2
        gross = (lr + sr) / 2
        rows.append({
            'date': dt,
            'long_return': lr,
            'short_return': sr,
            'gross_return': gross,
            'net_return': gross - cost,
            'turnover': to,
            'cost': cost
        })
        pl, ps = lt, st

    return pd.DataFrame(rows).sort_values('date').reset_index(drop=True)


# --- 检测信号方向 ---
print('Testing signal direction...')
for rev, lab in [(False, '原始'), (True, '反转')]:
    bt = backtest_from_predictions(res_final, rev=rev)
    r = bt['net_return']
    cum = (1 + r).cumprod()
    tot = cum.iloc[-1] - 1
    ppy = 252 / REBAL_FREQ
    ar = (1 + tot) ** (ppy / len(r)) - 1
    av = r.std() * np.sqrt(ppy)
    sh = (ar - 0.04) / av if av > 0 else 0
    print(f'  [{lab}] Net:{tot:.2%} | Ann:{ar:.2%} | Sharpe:{sh:.2f} | TO:{bt["turnover"].mean():.0%}')

bt_o = backtest_from_predictions(res_final, rev=False)
bt_r = backtest_from_predictions(res_final, rev=True)
use_rev = (1 + bt_r['net_return']).cumprod().iloc[-1] > (1 + bt_o['net_return']).cumprod().iloc[-1]
bt_final = bt_r if use_rev else bt_o
# 保留对应方向的 res_final 用于后续分析
res_plot = res_final.copy()
if use_rev:
    res_plot['predicted'] = -res_plot['predicted']
print(f'\n✅ Using {"reversed" if use_rev else "original"} signal | Trades: {len(bt_final)}')



## 8. 评估

In [ ]:
# =====================================================================
# Cell 8 替换：评估（不变，但加上 rolling window 的额外指标）
# =====================================================================

def evaluate(bt, rf=REBAL_FREQ):
    ppy = 252 / rf
    for lab, col in [('Gross (no cost)', 'gross_return'), ('Net (with cost)', 'net_return')]:
        r = bt[col]
        cum = (1 + r).cumprod()
        tot = cum.iloc[-1] - 1
        n = len(r)
        ar = (1 + tot) ** (ppy / n) - 1
        av = r.std() * np.sqrt(ppy)
        sh = (ar - 0.04) / av if av > 0 else 0
        md = ((cum - cum.cummax()) / cum.cummax()).min()
        wr = (r > 0).mean()
        print(f'\n  {"="*45}')
        print(f'  {lab}')
        print(f'  {"="*45}')
        print(f'    Total return:     {tot:.2%}')
        print(f'    Annualized return:{ar:.2%}')
        print(f'    Annualized vol:   {av:.2%}')
        print(f'    Sharpe ratio:     {sh:.2f}')
        print(f'    Max drawdown:     {md:.2%}')
        print(f'    Win rate:         {wr:.0%}')

    # 新增：rolling window 特有的稳定性指标
    print(f'\n  {"="*45}')
    print(f'  Rolling window stability')
    print(f'  {"="*45}')
    # 按年分组看年化收益
    bt_copy = bt.copy()
    bt_copy['year'] = pd.to_datetime(bt_copy['date']).dt.year
    for year, grp in bt_copy.groupby('year'):
        yr = grp['net_return']
        yr_tot = (1 + yr).cumprod().iloc[-1] - 1
        yr_sh = (yr.mean() / yr.std() * np.sqrt(ppy)) if yr.std() > 0 else 0
        print(f'    {year}: Return={yr_tot:.2%} | Sharpe={yr_sh:.2f} | Trades={len(yr)}')

evaluate(bt_final)


In [ ]:
def evaluate(bt, rf=REBAL_FREQ):
    ppy = 252/rf
    for lab, col in [('📊 Gross（无成本）','gross_return'),('📊 Net（含成本）','net_return')]:
        r = bt[col]; cum = (1+r).cumprod(); tot = cum.iloc[-1]-1; n = len(r)
        ar = (1+tot)**(ppy/n)-1; av = r.std()*np.sqrt(ppy)
        sh = (ar-0.04)/av if av>0 else 0; md = ((cum-cum.cummax())/cum.cummax()).min(); wr = (r>0).mean()
        print(f'\n  {"="*45}')
        print(f'  {lab}')
        print(f'  {"="*45}')
        print(f'    总收益率:     {tot:.2%}')
        print(f'    年化收益率:   {ar:.2%}')
        print(f'    年化波动率:   {av:.2%}')
        print(f'    夏普比率:     {sh:.2f}')
        print(f'    最大回撤:     {md:.2%}')
        print(f'    胜率:         {wr:.0%}')

evaluate(bt_final)

## 9. 可视化

In [ ]:
# =====================================================================
# Cell 9 替换：可视化（用 res_plot 代替 res_final）
# =====================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('V3 Quant Strategy (Rolling Window)', fontsize=14, fontweight='bold')

# 左上：累计收益 + fold 分割线
ax = axes[0, 0]
ax.plot(bt_final['date'], (1 + bt_final['gross_return']).cumprod(), 'b-', label='Gross', lw=1.5)
ax.plot(bt_final['date'], (1 + bt_final['net_return']).cumprod(), 'r--', label='Net', lw=1.5)
ax.axhline(y=1, color='gray', ls='--', alpha=0.5)
# 标注每个 fold 的分界线
for fm in fold_metrics:
    ax.axvline(x=fm['test_start'], color='green', ls=':', alpha=0.4, lw=0.8)
ax.legend()
ax.set_title('Cumulative returns (green lines = fold boundaries)')
ax.grid(True, alpha=0.3)

# 右上：Long vs Short 分布
ax = axes[0, 1]
ax.hist(bt_final['long_return'], bins=40, alpha=0.5, color='green', label='Long')
ax.hist(bt_final['short_return'], bins=40, alpha=0.5, color='red', label='Short')
ax.axvline(x=bt_final['long_return'].mean(), color='green', lw=2,
           label=f"Long={bt_final['long_return'].mean():.4f}")
ax.axvline(x=bt_final['short_return'].mean(), color='red', lw=2,
           label=f"Short={bt_final['short_return'].mean():.4f}")
ax.legend(fontsize=8)
ax.set_title('Long vs Short returns')
ax.grid(True, alpha=0.3)

# 左下：Pred vs Actual 散点图
ax = axes[1, 0]
s = res_plot.sample(min(3000, len(res_plot)))
ax.scatter(s['predicted'], s['actual'], alpha=0.1, s=5)
ax.axhline(y=0, color='gray', ls='--')
ax.axvline(x=0, color='gray', ls='--')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Pred vs Actual')
ax.grid(True, alpha=0.3)

# 右下：分位数收益
ax = axes[1, 1]
res_plot['q'] = pd.qcut(res_plot['predicted'], q=5,
                         labels=['Q1\n(Short)', 'Q2', 'Q3', 'Q4', 'Q5\n(Long)'])
qr = res_plot.groupby('q')['actual'].mean()
bars = ax.bar(qr.index, qr.values,
              color=['#e74c3c', '#e67e22', '#95a5a6', '#27ae60', '#2ecc71'],
              edgecolor='k', lw=0.5)
ax.axhline(y=0, color='k', ls='--', lw=0.5)
for b, v in zip(bars, qr.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.0002, f'{v:.4f}',
            ha='center', fontsize=9)
ax.set_title('Return by quantile')
ax.set_ylabel('Mean return')

plt.tight_layout()
plt.savefig('v3_rolling_results.png', dpi=150)
plt.show()


# =====================================================================
# Cell 10 替换：IC 分析（用 res_plot）
# =====================================================================

daily_ic = []
for dt, g in res_plot.groupby('date'):
    if len(g) < 5:
        continue
    ic, _ = stats.spearmanr(g['predicted'], g['actual'])
    if not np.isnan(ic):
        daily_ic.append({'date': dt, 'ic': ic})

idf = pd.DataFrame(daily_ic).sort_values('date')
mic, sic = idf['ic'].mean(), idf['ic'].std()
ir = mic / sic if sic > 0 else 0

print(f'Mean IC: {mic:.4f} | IC Std: {sic:.4f} | IR: {ir:.4f} | IC>0: {(idf["ic"] > 0).mean():.0%}')
if abs(mic) > 0.05:
    print('→ ✅ Good')
elif abs(mic) > 0.03:
    print('→ ⚠️ Weak')
else:
    print('→ ❌ Near random')

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

ax = axes[0]
ax.bar(range(len(idf)), idf['ic'],
       color=['g' if x > 0 else 'r' for x in idf['ic']], alpha=0.6, width=1)
ax.axhline(y=mic, color='blue', ls='--', label=f'Mean={mic:.4f}')
# fold 分界线
for fm in fold_metrics:
    idx_match = idf[idf['date'] >= fm['test_start']].index
    if len(idx_match) > 0:
        pos = list(idf.index).index(idx_match[0])
        ax.axvline(x=pos, color='green', ls=':', alpha=0.5, lw=0.8)
ax.legend()
ax.set_title('Daily IC (green lines = fold boundaries)')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(idf['ic'].cumsum().values, 'b-', lw=1.5)
ax.set_title('Cumulative IC')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 10. IC 分析

In [ ]:
# ====================================================================
# Cell 10.5（新增）：单股预测（适配 rolling window）
# =====================================================================

def plot_stock_prediction_rolling(ticker, res_plot):
    """直接用 res_plot 中的预测结果，不再需要 model"""
    mask = res_plot[res_plot['ticker'] == ticker].copy()
    if len(mask) == 0:
        print(f"❌ {ticker} not in predictions")
        print(f"Available: {sorted(res_plot['ticker'].unique())}")
        return

    mask = mask.sort_values('date')
    dates = mask['date'].values
    actuals = mask['actual'].values
    preds = mask['predicted'].values

    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    fig.suptitle(f'{ticker} — Rolling Window Predictions', fontsize=14, fontweight='bold')

    # 预测 vs 实际
    ax = axes[0]
    ax.plot(dates, actuals, 'b-', label='Actual', alpha=0.8, lw=1.5)
    ax.plot(dates, preds, 'r--', label='Predicted', alpha=0.8, lw=1.5)
    ax.axhline(y=0, color='gray', ls='--', lw=0.5)
    ax.fill_between(dates, actuals, preds, alpha=0.15, color='purple')
    for fm in fold_metrics:
        ax.axvline(x=fm['test_start'], color='green', ls=':', alpha=0.4)
    ax.legend(fontsize=10)
    ax.set_title('Predicted vs Actual (5-day return)')
    ax.grid(True, alpha=0.3)

    # 偏差
    ax = axes[1]
    dev = preds - actuals
    colors = ['green' if d >= 0 else 'red' for d in dev]
    ax.bar(dates, dev, color=colors, alpha=0.6, width=2)
    ax.axhline(y=0, color='black', lw=0.5)
    ax.axhline(y=dev.mean(), color='blue', ls='--', label=f'Mean dev: {dev.mean():.4f}')
    ax.legend()
    ax.set_title('Prediction deviation')
    ax.grid(True, alpha=0.3)

    # 方向准确率
    ax = axes[2]
    correct = (np.sign(preds) == np.sign(actuals)).astype(int)
    cum_acc = pd.Series(correct).expanding().mean()
    ax.plot(dates, cum_acc, 'purple', lw=2)
    ax.axhline(y=0.5, color='gray', ls='--', label='Random 50%')
    ax.set_ylim(0.3, 0.8)
    ax.legend()
    ax.set_title(f'Directional accuracy (final: {correct.mean():.1%})')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{ticker}_rolling_pred.png', dpi=150, bbox_inches='tight')
    plt.show()

    corr = np.corrcoef(preds, actuals)[0, 1]
    print(f"\n{'='*40}")
    print(f" {ticker} Prediction Stats (Rolling Window)")
    print(f"{'='*40}")
    print(f"  Samples:       {len(mask)}")
    print(f"  Date range:    {pd.Timestamp(dates[0]).strftime('%Y-%m-%d')} ~ {pd.Timestamp(dates[-1]).strftime('%Y-%m-%d')}")
    print(f"  Mean dev:      {dev.mean():.4f}")
    print(f"  Direction acc: {correct.mean():.1%}")
    print(f"  Correlation:   {corr:.4f}")

plot_stock_prediction_rolling('AAPL', res_plot)

In [ ]:
plot_stock_prediction_rolling('COP', res_plot)

## 11. 可调参数

| 参数 | 位置 | 当前值 | 试试 |
|------|------|--------|------|
| `d_model` | Cell 4 | 192 | 128 / 256 |
| `num_layers` | Cell 4 | 4 | 2 / 6 |
| `nhead` | Cell 4 | 6 | 3 / 8 |
| `dropout` | Cell 4 | 0.15 | 0.1 / 0.25 |
| `rank/mse/ic weight` | Cell 5 | 0.5/0.2/0.3 | 0.7/0.1/0.2 |
| `NUM_EPOCHS` | Cell 6 | 80 | 60 / 120 |
| `T_max` | Cell 6 | 40 | 30 / 50 |
| `lr` | Cell 6 | 3e-4 | 1e-4 / 5e-4 |
| `TOP_N` | Cell 7 | 3 | 2 / 5 |
| `REBAL_FREQ` | Cell 7 | 5 | 3 / 10 |
| `train_ratio` | Cell 3 | 0.70 | 0.75 / 0.80 |